# 🚦 Train YOLOv8n — Biển Báo Giao Thông Việt Nam

## Chuẩn bị trước khi chạy:
1. **Runtime → Change runtime type → T4 GPU** ✅
2. Upload thư mục `archive/` lên Google Drive tại đường dẫn:
   ```
   My Drive/traffic-sign-quantization/archive/
   ```
   Cấu trúc cần có:
   ```
   archive/
   ├── images/          ← tất cả ảnh .jpg
   ├── labels/          ← tất cả annotation .txt
   ├── classes.txt      ← 52 tên biển báo
   └── split_dataset/
       ├── train_files.txt
       └── test_files.txt
   ```

In [ ]:
# CELL 1 — Kiểm tra GPU và mount Google Drive
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO GPU — Doi runtime!"}')
from google.colab import drive
drive.mount('/content/drive')
print('Drive da mount!')

In [ ]:
# CELL 2 — Cai thu vien
!pip install ultralytics -q
print('Ultralytics da cai!')

In [ ]:
# CELL 3 — Thiet lap duong dan
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/traffic-sign-quantization')
ARCHIVE    = DRIVE_BASE / 'archive'
IMAGES_DIR = ARCHIVE / 'images'
LABELS_DIR = ARCHIVE / 'labels'
CLASSES_FILE  = ARCHIVE / 'classes.txt'
TRAIN_LIST = ARCHIVE / 'split_dataset' / 'train_files.txt'
TEST_LIST  = ARCHIVE / 'split_dataset' / 'test_files.txt'
WORK_DIR   = Path('/content/traffic_sign_yolo')
WORK_DIR.mkdir(exist_ok=True)

print(f'Images: {IMAGES_DIR.exists()}')
print(f'Labels: {LABELS_DIR.exists()}')
print(f'Classes: {CLASSES_FILE.exists()}')
print(f'Train list: {TRAIN_LIST.exists()}')
print(f'Test list:  {TEST_LIST.exists()}')

In [ ]:
# CELL 4 — Tao file paths tuyet doi cho Colab
def make_path_file(list_file, images_dir, out_file):
    with open(list_file) as f:
        fnames = [l.strip() for l in f if l.strip()]
    valid = []
    for fname in fnames:
        img = images_dir / fname
        lbl = Path(str(img).replace('/images/', '/labels/').replace('.jpg', '.txt'))
        if img.exists() and lbl.exists():
            valid.append(str(img.resolve()))
    with open(out_file, 'w') as f:
        f.write('\n'.join(valid))
    print(f'{out_file.name}: {len(valid)} anh hop le')
    return out_file

TRAIN_TXT = WORK_DIR / 'train_paths.txt'
VAL_TXT   = WORK_DIR / 'val_paths.txt'
make_path_file(TRAIN_LIST, IMAGES_DIR, TRAIN_TXT)
make_path_file(TEST_LIST,  IMAGES_DIR, VAL_TXT)

In [ ]:
# CELL 5 — Tao YAML dataset config
with open(CLASSES_FILE) as f:
    class_names = [l.strip() for l in f if l.strip()]
print(f'So lop: {len(class_names)} | Vi du: {class_names[:5]}')

yaml_content = f'train: {TRAIN_TXT}\nval:   {VAL_TXT}\nnc: {len(class_names)}\nnames:\n'
for i, name in enumerate(class_names):
    yaml_content += f'  {i}: {name}\n'

YAML_FILE = WORK_DIR / 'traffic_signs_colab.yaml'
YAML_FILE.write_text(yaml_content)
print(f'YAML tao xong: {YAML_FILE}')

In [ ]:
# CELL 6 — TRAIN YOLOv8n (~10-20 phut tren T4 GPU)
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=str(YAML_FILE),
    epochs=50,
    imgsz=640,
    batch=32,
    patience=15,
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    mosaic=0.8,
    fliplr=0.5,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    device=0,
    project='/content/runs',
    name='baseline',
    exist_ok=True,
)

print('='*50)
print('TRAINING HOAN TAT!')
print(f'mAP50     : {results.results_dict["metrics/mAP50(B)"]:.4f}')
print(f'mAP50-95  : {results.results_dict["metrics/mAP50-95(B)"]:.4f}')
print(f'Precision : {results.results_dict["metrics/precision(B)"]:.4f}')
print(f'Recall    : {results.results_dict["metrics/recall(B)"]:.4f}')

In [ ]:
# CELL 7 — Luu model ve Google Drive
import shutil
best_pt = Path('/content/runs/baseline/weights/best.pt')
output_dir = DRIVE_BASE / 'checkpoints'
output_dir.mkdir(parents=True, exist_ok=True)
dest = output_dir / 'baseline.pt'
shutil.copy(best_pt, dest)
print(f'Da luu: {dest}')
print(f'Kich thuoc: {dest.stat().st_size / 1024**2:.2f} MB')
print('\nBuoc tiep theo:')
print('  Vao Google Drive -> traffic-sign-quantization/checkpoints/')
print('  Download baseline.pt ve Mac')
print('  Dat vao: .../traffic-sign-quantization/checkpoints/baseline.pt')

In [ ]:
# CELL 8 (Tuy chon) — Download thang ve may
from google.colab import files
files.download('/content/runs/baseline/weights/best.pt')
print('Dang download baseline.pt ve may...')